# Project 1

Welcome to the first NLP1 project. The goal of the first tutorial was to get you familiar with text segmentation and simple text classification for topic detection.
Given the skills you have developed in the first week, you will now be applying them within a different dataset and task.

We are recommending a set of tasks and associated datasets; using those, you will be experimenting on problems introduced during the tutorials. You will be applying hands-on the experiments you have already explored in the last tutorial, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and analysing errors. 

The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

As the tutorials will be building upon your knowledge throughout the weeks, the task and dataset you choose now will be the ones you use in projects 2 and 3; and the ones you will be finally writing about in your final blog post.  

## Get to work!

The next section has a checklist to help you not lose track of what needs to be done. Here we have a more informal guide, with prompts and questions for you. 


1. [Choose a dataset](http://nlp-illc.github.io/nlp1/). Spend some time understanding how the dataset looks and some statistics of it (e.g. how many instances of each classification label we have?)
2. Remember what we learnt during the tutorial: to develop any classifier of the kind we saw, we needed a way to segment text. Do you want to use the tokeniser we have developed? Do you need to train a new one? If you do, what hyperparameters shall you use? What data shall you use to train it? Consider that your dataset may be too small to train a tokeniser! Think whether you will need a larger resource of similar nature to train a tokeniser.
3. Start thinking how to represent your documents into features. We have experimented with token occurences and frequencies. Can you think of any other ways, possibly? 
4. We had experimented with Logistic Regression and a shallow neural network. You can experiment with and assess different architectures now and their hyper parameters (e.g. add layers, possibly change the activation functions, drop out rates).
5. Evaluate your models! Was the evaluation method adequate for your new experimental setup? Do you want to assess other aspects of performance?
6. Analyse the various models you assessed and try to decide on which is the best.

Be mindful that you will be using your experiments to write up your final blog, so make sure they are clean and neat!

You can write your own code and you can reuse code from the tutorial and its packages or from other sources. You may use coding assistants if you like, but use it mindfully as you take full responsibility for this kind of (semi-)automated coding. Also, keep a record of the kinds of things you delegate to any form of assistant, as you will be asked to disclose this in your blog post (not because we will pass a value judgement on it, but because it is a widespread and good practice to document such things).

You may use jupyter notebooks or python files, it's up to you as we will not directly assess your code. But, once again, be organised, as we will ask you to share a reproducible version of your experiments along with your blog post (again, because this is how real research is publicised). 

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

## Checklist

What **we expect you to do**:

- [ ] Compare different systems of text classification for your choice of data/task.
- Your comparisons should cover:
    - [ ] the two representation spaces in the tutorial (token indicators and bag-of-words) and an additional one of your choice.
    - [ ] the two kinds of architectures in the tutorial (log-linear and single-hidden-layer MLP) and an additional one of your choice (such as a modification of the MLP).
- [ ] Your comparisons must be based on cross-validation and a relevant metric (accuracy or else).
- [ ] You should make use of Bayesian estimation to help you analyse results in relation to a region of practical equivalence (ROPE), and you can motivate the ROPE size yourself.

What you need to **bring to the oral assessment**:
* You will be asked questions about the points above, you can bring a slide deck with up to five slides (one for each of the following topics _data and metrics_, _features_, _architectures_, _results_, _extra_); you _will not_ give a presentation, rather, these slides are to be thought of as "backup slides" that you can use to guide your answers to the questions the TA will ask you.
* You may also be asked questions about the tutorial, for those, you don't need to prepare any deliverable (a good way to prepare is to work through the tutorial and reflect on the questions there).

What you should **keep in mind** given that you will eventually write a blog post about your work throughout the course:
* Organise your code;
* Summarise your motivations and findings;
* Document your sources and any form of AI-assistance you use.

---
# Our project: Name that LLM

## 0. Reproducibility

Every experiment below is seeded so anyone can re-run it and get the same numbers.

- **During development** use one seed (`MODE = "dev"`), so iterations stay fast.
- **For reported results** switch to `MODE = "report"`: every experiment runs once per seed (5 seeds × k folds). We report the mean and a 95% interval across runs. `BenavoliCorrelatedT(..., runs=len(SEEDS))` in `part1.analysis` accounts for the fold overlap.
- Call `set_seed(seed)` **right before** every split / training call. `part1.smlp.train_neural_network_model` does not seed PyTorch itself, so without this the MLP weights and batch order change on every run.
- The seed controls the fold split (`cross_validation_split(..., seed=seed)`) **and** the model's initialisation, so each run is a different split + initialisation.

In [ ]:
# ---- Reproducibility setup: run this cell first ----
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import torch

warnings.simplefilter(action="ignore", category=FutureWarning)

# Paths (this notebook lives in root/part1)
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import part1  # noqa: E402  (course helpers: experiment, loglin, smlp, analysis, ...)

DATA_DIR = ROOT / "data" / "name-that-llm"
GRAPHS_DIR = ROOT / "graphs"
GRAPHS_DIR.mkdir(exist_ok=True)
RESULTS_DIR = ROOT / "outputs" / "part1"  # gitignored
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# Seeds
MODE = "dev"  # "dev" -> 1 seed (fast) | "report" -> 5 seeds (for the final numbers)
DEV_SEEDS = [0]
REPORT_SEEDS = [0, 1, 2, 3, 4]
SEEDS = DEV_SEEDS if MODE == "dev" else REPORT_SEEDS
NUM_FOLDS = 5
SPLIT_SEED = 42


def set_seed(seed: int) -> None:
    """Seed every RNG we use (Python, NumPy, PyTorch) and make torch deterministic where possible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(
        seed
    )  # only affects subprocesses; kernel hash seed is fixed at startup
    torch.use_deterministic_algorithms(True, warn_only=True)


def summarise_runs(scores, confidence=0.95):
    """Mean and t-based CI over independent runs (e.g. one mean accuracy per seed)."""
    from scipy import stats

    scores = np.asarray(scores, dtype=float)
    mean = scores.mean()
    if len(scores) < 2:
        return mean, (np.nan, np.nan)
    half = stats.t.ppf((1 + confidence) / 2, len(scores) - 1) * stats.sem(scores)
    return mean, (mean - half, mean + half)


set_seed(SEEDS[0])
print(
    f"MODE={MODE} | SEEDS={SEEDS} | folds={NUM_FOLDS} | torch {torch.__version__} | numpy {np.__version__}"
)

## 1. Data: Name that LLM

**What they ask:** choose a dataset, spend time understanding it and compute some statistics (e.g. how many instances of each label).

**Our task:** given a text, predict which LLM (`model` column) wrote it. Kaggle competition: [name-that-llm](https://www.kaggle.com/competitions/name-that-llm).

**Rubric / oral-assessment points** (slide: _data and metrics_):
- How many classes, how many examples per class? Is it balanced, and what does that imply for the metric (accuracy vs. macro-F1)?
- What is `prompt_id`? The same prompt is answered by several models, so folds should be grouped by prompt to avoid leakage (see Step 5).
- Text length per model: could a classifier "cheat" on length alone?
- Which models are near-duplicates (same family, e.g. `gemini-3.x-flash`, `gpt-5.6-*`)? Expect confusions there.
- `test.csv` only has a handful of rows (the real Kaggle test set is hidden), so all evaluation is via cross-validation on `train.csv`.

### Data overview

Load `train.csv`, count responses per model, check how many models answer each prompt and look at length per model. Also builds `outcome_space` (label ↔ id) and `pairs` for the course helpers.

**Takeaway:** 18 balanced classes; 150 prompts, each answered by all 18 models, except one missing response (prompt `p002`, claude-fable-5).

In [ ]:
import pandas as pd
from IPython.display import display

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
print(train_df.shape, test_df.shape)
display(train_df.head())

# Label distribution
display(train_df["model"].value_counts().to_frame("count"))

# Prompts: how many, and how many models answer each one?
print("unique prompts:", train_df["prompt_id"].nunique())
display(train_df.groupby("prompt_id")["model"].nunique().describe())

# Length per model (characters / whitespace tokens)
train_df["n_chars"] = train_df["text"].str.len()
train_df["n_words"] = train_df["text"].str.split().str.len()
display(train_df.groupby("model")[["n_chars", "n_words"]].mean().sort_values("n_words"))

# Outcome space: map label names <-> integer ids (as in the tutorial)
outcome_space = part1.clslib.NamedClasses(sorted(train_df["model"].unique()))
pairs = [(t, outcome_space.index(m)) for t, m in zip(train_df["text"], train_df["model"])]
print(len(outcome_space), "classes")

### Data splitting

80/20 split by prompt (`GroupShuffleSplit`, `groups=prompt_id`, fixed `SPLIT_SEED`): 120 prompts (2,160 rows) for training and 30 prompts (540 rows) held out, so no prompt appears on both sides. The held-out set stays untouched until final evaluation; all EDA uses `train_split_df` only.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=SPLIT_SEED)
train_idx, test_idx = next(gss.split(train_df, y=train_df["model"], groups=train_df["prompt_id"]))

train_split_df = train_df.iloc[train_idx].reset_index(drop=True)
test_split_df = train_df.iloc[test_idx].reset_index(drop=True)

train_split_df.to_csv(DATA_DIR / "split_train.csv", index=False)
test_split_df.to_csv(DATA_DIR / "split_test.csv", index=False)

# Verify the split is correct
assert train_split_df["prompt_id"].nunique() == 0.8 * train_df["prompt_id"].nunique()
assert test_split_df["prompt_id"].nunique() == 0.2 * train_df["prompt_id"].nunique()
assert set(train_split_df["prompt_id"]).isdisjoint(test_split_df["prompt_id"])

print(len(train_split_df), len(test_split_df))

### Model families

Maps each of the 18 models to its company (6 families: Anthropic 5, OpenAI 5, Google 4, GLM 2, Kimi 1, Qwen 1). The EDA uses it to ask whether a difference between models is a family habit or an individual-model habit.

In [ ]:
OpenAI = "OpenAI"
Llama = "Llama"
Google = "Google"
Anthropic = "Anthropic"
Qwen = "Qwen"
Glm = "GLM"
Kimi = "Kimi"

model_family_map = {
    "qwen3.8-max": Qwen,
    "gemini-3.1-pro-preview": Google,
    "gemini-3.7-flash": Google,
    "gemini-3.6-flash": Google,
    "gemini-3.8-flash": Google,
    "gpt-5.6-terra": OpenAI,
    "gpt-5.6-sol": OpenAI,
    "gpt-5.5": OpenAI,
    "gpt-5.6-luna": OpenAI,
    "gpt-6-astra": OpenAI,
    "claude-opus-5": Anthropic,
    "claude-sonnet-5": Anthropic,
    "claude-opus-4-6": Anthropic,
    "claude-opus-4-8": Anthropic,
    "claude-fable-5": Anthropic,
    "glm-5.3": Glm,
    "glm-5.3-flash": Glm,
    "kimi-k3": Kimi,
}

### EDA

All EDA runs on the training split only. Every model answers every prompt, so features are compared **within prompts**: each response is ranked among the 18 answers to the same prompt (0 = lowest, 1 = highest), which removes prompt-to-prompt variation such as a one-line answer vs. an essay. Prompt `p002` is missing one response and is left out of these paired tests (119 prompts).

Sections:

1. Response length and word length
2. Punctuation and quote style
3. Distinctive vocabulary and vocabulary rates
4. Model similarity (TF-IDF cosine within prompts)
5. Wrap-up: final feature table, correlations and the kept feature set

Helpers live in `part1/eda.py`; plots are saved to `graphs/eda/`.

#### Response length and word length

Could a classifier tell models apart by how much they write, or by how long their words are? The first cell is a quick look with plain pandas; the cells after it redo it with the `eda.py` helpers: summary statistics per model, distributions by family and by model, and within-prompt ranks (`analyse_feature`).

**Takeaway:** length carries no signal (every model writes ~35–275 words, which looks controlled by design); word length does differ between models (W = 0.35).

In [ ]:
eda_df = train_split_df.copy()
eda_df["family"] = eda_df["model"].map(model_family_map)
text = eda_df["text"]

eda_df["n_chars"] = text.str.len()
eda_df["n_words"] = text.str.split().str.len()
eda_df["chars_per_word"] = eda_df["n_chars"] / eda_df["n_words"].replace(0, np.nan)

stats = (
    eda_df.groupby(["family", "model"])[["n_words", "n_chars", "chars_per_word"]]
    .agg(["mean", "median", "std"])
    .round(2)
)

display(stats)

family_stats = (
    eda_df.groupby("family")[["n_words", "n_chars", "chars_per_word"]]
    .agg(["mean", "median", "std"])
    .round(2)
)

display(family_stats)

In [ ]:
from eda import add_length_features, analyse_feature, summarise_by_group

EDA_GRAPHS_DIR = GRAPHS_DIR / "eda"
EDA_GRAPHS_DIR.mkdir(exist_ok=True)

In [ ]:
eda_df = add_length_features(train_split_df)
eda_df["family"] = eda_df["model"].map(model_family_map)

display(summarise_by_group(eda_df, ["n_words", "n_chars", "chars_per_word"]))

In [ ]:
res_len = analyse_feature(
    eda_df,
    "n_words",
    "Words per response",
    save_dir=EDA_GRAPHS_DIR,
    prefix="resp_len",
    log_scale_model=True,
)

In [ ]:
res_cpw = analyse_feature(
    eda_df,
    "chars_per_word",
    "Characters per word",
    save_dir=EDA_GRAPHS_DIR,
    prefix="chars_per_word",
)

**Tests** (`feature_tests`, reused for every numeric feature below). For each feature:

1. *Do models differ?* Friedman test with prompts as blocks and the 18 models as treatments; effect size Kendall's W (0 = models ordered randomly from prompt to prompt, 1 = same order on every prompt).
2. *Do the family averages differ?* The same test on prompts × 6 families, using each family's mean rank per prompt.
3. *Does family membership matter?* Permutation test: η² is the share of the variance in the models' mean ranks that lies between families, compared with shuffled family labels. Chance η² is about 0.30, not 0.

`letters_per_word` strips punctuation from each token, so it measures word choice rather than punctuation.

In [ ]:
from eda import feature_tests, style_feature_tests

tests = feature_tests(eda_df, ["n_words", "chars_per_word", "letters_per_word"])
display(style_feature_tests(tests))

#### Punctuation

Each punctuation mark is counted per 100 words (`add_punctuation_features`) and averaged per model. The heatmap colours each column by its own range, so rare and common marks are equally readable; the numbers are the raw rates.

**Takeaway:** punctuation is mostly a family habit. Em-dash rate is the strongest single feature (W = 0.48), and Google barely uses em-dashes, colons or hyphens.

In [ ]:
from part1.eda import (
    PUNCT_MARKS,
    QUOTE_MARKS,
    add_length_features,
    add_punctuation_features,
    feature_tests,
    plot_model_heatmap,
    punct_cols,
    quote_style_table,
    quote_style_test,
    style_feature_tests,
    usage_share,
)

eda_df = add_length_features(train_split_df)
eda_df["family"] = eda_df["model"].map(model_family_map)
eda_df = add_punctuation_features(eda_df)

rates = eda_df.groupby(["family", "model"])[punct_cols()].mean()
plot_model_heatmap(
    rates,
    labels={c: c.removesuffix("_per100") for c in punct_cols()},
    title="Punctuation marks per 100 words (mean per response)",
    save_path=EDA_GRAPHS_DIR / "punct_per100_heatmap.png",
)

#### Quote style

Each response containing a quote character is classed as curly (’ “ ”), straight (' ") or mixed. Shares are per model, over responses with quotes only. A chi-square test with Cramér's V measures how strongly style depends on the model (p is descriptive only, since answers to the same prompt are not independent). The last table is the share of responses that use each quote character at all.

**Takeaway:** Anthropic and GLM never use curly quotes, and OpenAI mostly does (Cramér's V = 0.61). Preprocessing must not normalise quotes.

In [ ]:
qt = quote_style_table(eda_df)
plot_model_heatmap(
    qt[["curly", "straight", "mixed"]],
    scale="global",
    cmap="Purples",
    title="Quote style (responses containing quotes)",
    cbar_label="share of responses",
    save_path=EDA_GRAPHS_DIR / "quote_style_heatmap.png",
)
display(qt)
print(quote_style_test(eda_df))
display(usage_share(eda_df, [f"n_{q}" for q in QUOTE_MARKS]).round(2))

Punctuation rates through the three tests above, sorted by W. Marks that never occur (spaced hyphen, `…`) give empty rows.

In [ ]:
punct_tests = feature_tests(eda_df, punct_cols())
display(style_feature_tests(punct_tests))

#### Distinctive vocabulary

Which words does a model (or family) use more than the others? Log-odds ratio with an informative Dirichlet prior (Monroe et al., 2008): a z-score per term, with rare terms shrunk towards zero. Topic words cancel out because every model answers the same prompts. Curly and straight apostrophes are merged here, so quote style doesn't split words.

Cells, in order: each family vs. the rest; each model vs. the rest (table, then plot); each model vs. the rest of its own family; bigrams.

**Takeaway:** only Google has a strong vocabulary signature (*profound*, *an* + Latinate adjectives, *within the realm of*); the other families differ mostly in function words.

In [ ]:
from eda import (
    distinctive_terms,
    make_palette,
    plot_distinctive_terms,
    top_terms_summary,
)

model_to_family = dict(zip(eda_df["model"], eda_df["family"]))
pal = make_palette(eda_df["family"])

fam_terms = distinctive_terms(eda_df, group_col="family", k=15)
display(top_terms_summary(fam_terms))

In [ ]:
model_terms = distinctive_terms(eda_df, k=10)
display(top_terms_summary(model_terms))

In [ ]:
plot_distinctive_terms(
    model_terms,
    k=10,
    palette=pal,
    group_to_colour=model_to_family,
    title="Most distinctive words per model (vs. all other models)",
    save_path=EDA_GRAPHS_DIR / "distinctive_terms_by_model.png",
);

In [ ]:
within_terms = distinctive_terms(eda_df, within="family", k=10)
display(top_terms_summary(within_terms))

In [ ]:
display(top_terms_summary(distinctive_terms(eda_df, ngram_range=(2, 2), k=10)))

With about 5,000 terms tested, z ≈ 2–4 happens by chance, and many top terms come from a single prompt. Keep only terms with z ≥ 4 that appear in at least 15% of the model's responses.

In [ ]:
robust = model_terms[(model_terms["z"] >= 4) & (model_terms["share_responses"] >= 0.15)]
display(
    robust.sort_values(["group", "z"], ascending=[True, False])[
        ["group", "term", "z", "share_responses", "per10k", "per10k_rest"]
    ].round(2)
)

#### Vocabulary rates

The patterns from the word lists, turned into per-100-word features (`add_vocab_features`): *of / an*, hedging modals, contractions, second-person pronouns, digits and number words. They then go through the same three tests.

**Takeaway:** *of / an* (Google) and hedging (OpenAI) are about as strong as colon and comma rates and are family habits; second person is the only one that varies mainly between models.

In [ ]:
from eda import add_vocab_features, vocab_cols

# Rates per 100 words of contractions, hedges, of/an, second person, digits and number words
eda_df = add_vocab_features(eda_df)
vocab_tests = feature_tests(eda_df, vocab_cols())
display(style_feature_tests(vocab_tests))

#### Model similarity: TF-IDF cosine within prompts

Are models from the same family more similar to each other than to models from other families? For every prompt answered by all 18 models we compute the cosine similarity between their TF-IDF vectors (word unigrams, one vectoriser fitted on the whole training split), then average the 18 x 18 matrices over the 119 complete prompts. Comparing only responses to the same prompt keeps topic out of the comparison. The family question is a permutation test on the family labels: mean same-family similarity minus mean different-family similarity, with family labels shuffled across the 18 models (rows and columns move together).

**Takeaway:** same-family pairs are more similar (0.286 vs. 0.234, p < 0.001, and on all 119 prompts); GLM sits inside the Anthropic cluster, and Google is the most separate family.

In [ ]:
from eda import (
    double_centre,
    family_similarity_permutation_test,
    family_similarity_table,
    nearest_neighbours,
    plot_similarity_heatmap,
    within_prompt_similarity,
)

sim, sim_per_prompt = within_prompt_similarity(eda_df, analyzer="word")
print(f"{sim_per_prompt.shape[0]} prompts")

plot_similarity_heatmap(
    sim,
    model_to_family,
    palette=pal,
    title="Mean within-prompt TF-IDF cosine similarity (word unigrams), clustered",
    save_path=EDA_GRAPHS_DIR / "similarity_word_clustered.png",
)
plot_similarity_heatmap(
    sim,
    model_to_family,
    palette=pal,
    cluster=False,
    title="Mean within-prompt TF-IDF cosine similarity (word unigrams), family blocks",
    save_path=EDA_GRAPHS_DIR / "similarity_word_family.png",
);

In [ ]:
sim_overall, sim_per_family = family_similarity_permutation_test(sim, model_to_family)
print({k: round(v, 4) if isinstance(v, float) else v for k, v in sim_overall.items()})
display(sim_per_family.round(4))
display(family_similarity_table(sim, model_to_family).round(3))
display(nearest_neighbours(sim, model_to_family).round(3))

In [ ]:
# Robustness: other representations, per-prompt IDF, removing per-model 'hub' effects,
# and how consistent the within > between gap is across prompts
sim_char, _ = within_prompt_similarity(eda_df, analyzer="char")
sim_prompt_idf, _ = within_prompt_similarity(eda_df, analyzer="word", idf="prompt")
variants = {
    "word, global IDF": sim,
    "char 2-4-grams, global IDF": sim_char,
    "word, per-prompt IDF": sim_prompt_idf,
    "word, double-centred": double_centre(sim),
}
rows = {}
for name, s in variants.items():
    ov, fam_tab = family_similarity_permutation_test(s, model_to_family)
    nn = nearest_neighbours(s, model_to_family)
    rows[name] = {
        "within": ov["within_mean"],
        "between": ov["between_mean"],
        "diff": ov["diff"],
        "null_sd": ov["diff_null_sd"],
        "p": ov["p"],
        **{f"p_{f}": p for f, p in fam_tab["p"].items()},
        "nn_same_family": f"{nn['same_family'].sum()}/{len(nn)}",
    }
display(pd.DataFrame(rows).T)

iu = np.triu_indices(len(sim), k=1)
fams = sim.index.map(model_to_family).to_numpy()
same = fams[iu[0]] == fams[iu[1]]
gap = np.array([m[iu][same].mean() - m[iu][~same].mean() for m in sim_per_prompt])
print(f"within > between in {(gap > 0).sum()}/{len(gap)} prompts (median gap {np.median(gap):.3f})")
print(
    "corr. of pair similarities, word vs char:",
    round(np.corrcoef(sim.to_numpy()[iu], sim_char.to_numpy()[iu])[0, 1], 3),
)

plot_similarity_heatmap(
    sim_char,
    model_to_family,
    palette=pal,
    title="Mean within-prompt TF-IDF cosine similarity (char 2-4-grams), clustered",
    save_path=EDA_GRAPHS_DIR / "similarity_char_clustered.png",
);

#### Wrap-up: final feature table and feature correlations

1. **One final table.** All numeric features go through `feature_tests` in a single call, with a Holm correction across features (`adjust="holm"`), so the overview table in the report comes from one reproducible cell.
2. **Correlations.** Each feature is replaced by its within-prompt rank, so prompt length or topic cannot create correlation. *Response level* asks whether two features move together in individual answers (redundant inputs for a classifier). *Model level* asks whether they rank the 18 models the same way (shared signature), and a PCA of the model profiles shows how many independent directions there are.
3. **Kept features.** Drop features with no model-level signal (Holm p ≥ 0.05), then, for each response-level pair with |r| ≥ 0.7, the one with the lower W.

**Takeaway:** 14 features kept (`KEEP_FEATURES`). They are nearly uncorrelated within responses (median |r| = 0.06), but across models two components (Google vs. the rest, Anthropic + GLM vs. OpenAI + Qwen) explain 68% of the variation.

In [ ]:
from eda import (
    add_vocab_features,
    correlated_pairs,
    feature_correlations,
    model_profile_pca,
    plot_feature_correlations,
    plot_model_profiles,
    vocab_cols,
)

wrap_df = add_length_features(train_split_df)
wrap_df["family"] = wrap_df["model"].map(model_family_map)
wrap_df = add_vocab_features(add_punctuation_features(wrap_df))

candidates = ["n_words", "chars_per_word", "letters_per_word"] + punct_cols() + vocab_cols()
never_used = [c for c in candidates if (wrap_df[c].fillna(0) == 0).all()]
ALL_FEATURES = [c for c in candidates if c not in never_used]
print("never occur (left out):", never_used)

final_tests = feature_tests(wrap_df, ALL_FEATURES, adjust="holm")
final_tests.drop(columns=[("info", "dropped")]).to_csv(RESULTS_DIR / "eda_final_feature_tests.csv")
display(style_feature_tests(final_tests))

In [ ]:
# Features with model-level signal (Holm-adjusted Friedman p < 0.05)
SIGNAL_FEATURES = final_tests.index[final_tests[("model", "sig")]].tolist()
short = {c: c.removesuffix("_per100") for c in ALL_FEATURES}

corr_resp = feature_correlations(wrap_df, SIGNAL_FEATURES, level="response")
corr_model = feature_correlations(wrap_df, SIGNAL_FEATURES, level="model")
plot_feature_correlations(
    corr_resp,
    labels=short,
    title="Feature correlations, response level (within-prompt ranks)",
    save_path=EDA_GRAPHS_DIR / "feature_corr_response.png",
)
plot_feature_correlations(
    corr_model,
    labels=short,
    title="Feature correlations, model level (18 models' mean ranks)",
    save_path=EDA_GRAPHS_DIR / "feature_corr_model.png",
)
print("Response level, |r| >= 0.5:")
display(correlated_pairs(corr_resp, 0.5).round(3))
print("Model level, |r| >= 0.7:")
display(correlated_pairs(corr_model, 0.7).round(3))

In [ ]:
# Selection rule: (1) model-level signal after Holm; (2) for each response-level pair with
# |r| >= 0.7, drop the feature with the lower model-level W
W = final_tests[("model", "W")]
dropped = {
    c: "no model-level signal (Holm p >= 0.05)" for c in ALL_FEATURES if c not in SIGNAL_FEATURES
}
for _, row in correlated_pairs(corr_resp, 0.7).iterrows():
    a, b = row["feature_a"], row["feature_b"]
    weaker, stronger = (a, b) if W[a] < W[b] else (b, a)
    dropped.setdefault(weaker, f"r = {row['r']:.2f} with {stronger} (higher W)")
KEEP_FEATURES = [c for c in W.index if c not in dropped]  # sorted by W

display(pd.Series(dropped, name="reason").to_frame())
print(f"KEEP_FEATURES ({len(KEEP_FEATURES)}):", KEEP_FEATURES)

pca = model_profile_pca(wrap_df, KEEP_FEATURES)
display(pca["explained"].head(5).round(3).to_frame("share of variance"))
display(pca["loadings"].round(2).sort_values("PC1"))
plot_model_profiles(
    pca,
    model_to_family,
    palette=pal,
    title="Model profiles on the kept features (PCA of mean within-prompt ranks)",
    save_path=EDA_GRAPHS_DIR / "model_profiles_pca.png",
);

### Reflections — Step 1

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 2. Tokenisation

**What they ask:** to build any classifier we need a way to segment text. Do we reuse the tutorial's tokeniser, or train a new one? If new, with which hyperparameters (vocab size, min frequency, pre-tokeniser), and on which data? Our dataset may be too small to train a tokeniser, so would a larger resource of similar nature be needed?

**Rubric / oral-assessment points** (slide: _features_; tutorial questions may come up too):
- Why BPE (subwords) vs. whitespace / character tokenisation for _authorship_ of LLM text? Punctuation, formatting and casing may carry the stylistic signal, so should we keep them?
- Vocab size trade-off: larger vocabularies give more specific features but sparser counts on ~2.7k documents.
- Training data for the tokeniser: train on all of `train.csv` (risk: it sees validation folds), only on the training folds, or an external corpus? Justify the choice.
- Unknown tokens / coverage: what fraction of tokens become `[UNK]`?

Helpers: `part1.experiment.train_tokenizer(corpus, vocab_size, min_frequency)`, `part1.bpe` (tutorial's BPE).

### Setup: data, folds and the three tokenisers

Step 2 picks, motivates and evaluates three tokenisers; training classifiers on them comes later.

- **Ours**: character-level BPE trained on our training data (course `train_tokenizer`, wrapped in `part1/tok.py`), with punctuation isolated so typography cues (— ’ “ ” ; :) stay separate tokens. Small, in-domain, readable tokens.
- **`roberta-bpe`**: the tokeniser of LUSR, a model trained to represent writing style and authorship (~50k byte-level BPE). It is also the neural track's vocabulary.
- **`qwen3-bpe`**: the ~150k byte-level BPE of Qwen3, a recent multilingual LLM family, used for the Qwen3 hidden-state features.

All tokenisers see the same preprocessed text and are wrapped in `tok.TokenizerView`, so the metrics treat them identically. Data and folds come from `shared/`: CV uses repeat 0 of `shared/folds.json` (5 folds grouped by prompt). The held-out split is not touched in this section.

In [ ]:
import tok

from shared import eval_harness as H
from shared.data import load_data
from shared.tokenization import PROBE

TOK_GRAPHS_DIR = GRAPHS_DIR / "tok"
TOK_GRAPHS_DIR.mkdir(exist_ok=True)

D = load_data()
# Same split as the EDA (load_data reads split_train.csv / split_test.csv, never re-splits)
assert D.train["id"].tolist() == pd.read_csv(DATA_DIR / "split_train.csv")["id"].tolist()

FOLDS = H.load_folds(D.train["id"].tolist(), repeats=1)  # repeat 0: 5 folds grouped by prompt
print(f"{len(D.train)} training responses, {D.train['prompt_id'].nunique()} prompts")
for r, k, tr, va in FOLDS:
    print(
        f"fold {k}: train {len(tr)} | validation {len(va)} ({D.train['prompt_id'].iloc[va].nunique()} prompts)"
    )

#### Preprocessing (identical for all tokenisers)

`shared.tokenization.preprocess` applies Unicode NFC normalisation and nothing else: no lowercasing (case may be style), no punctuation or stopword removal, no quote or dash mapping, no whitespace changes. The check below counts how many training texts NFC and NFKC would change. NFC only fixes how accented letters are encoded; NFKC also rewrites typographic characters, which are authorship cues, so we don't use it.

Pre-tokenisation (Metaspace vs. byte-level splitting) belongs to each tokeniser, not to preprocessing, and is one of the properties we compare.

In [ ]:
import unicodedata

train_texts = D.train["text"]
nfc_changes = train_texts.map(lambda t: unicodedata.normalize("NFC", t) != t)
nfkc_changes = train_texts.map(lambda t: unicodedata.normalize("NFKC", t) != t)
print(
    f"NFC changes {nfc_changes.sum()} of {len(train_texts)} training texts; "
    f"NFKC would change {nfkc_changes.sum()}"
)

rewritten = sorted(
    {c for t in train_texts[nfkc_changes] for c in t if unicodedata.normalize("NFKC", c) != c}
)
print(
    "characters NFKC would rewrite:",
    " ".join(f"{c} -> {unicodedata.normalize('NFKC', c)}" for c in rewritten),
)

#### Pretrained tokenisers

Loaded through the shared registry (`shared.tokenization.get_tokenizer`; downloaded from Hugging Face on first use). The table shows the vocabulary size, the pipeline components and how each tokeniser handles `PROBE`, a test string with curly quotes, an em-dash, `µ`, `…` and a double space (`probe_roundtrip_ok`: encoding then decoding gives the text back unchanged).

In [ ]:
pretrained = [tok.TokenizerView.from_shared(name) for name in tok.PRETRAINED]
display(tok.describe(pretrained))

#### Our BPE

`tok.train_bpe` wraps the course's `train_tokenizer` (HF `tokenizers` BPE with `[UNK]`, alphabet = every character in the training texts, no normaliser) and changes only the pre-tokeniser: Metaspace followed by isolating each punctuation character, so BPE never merges punctuation into words. It stays character-level (no byte-level fallback), so tokens are readable and UNK remains a meaningful metric.

Settings for the evaluation: vocab sizes `tok.VOCAB_SIZES` = {500, 1k, 2k, 4k, 8k}, `min_frequency = 2`, and punctuation isolated vs. vanilla Metaspace as an ablation. The evaluation trains one tokeniser per CV fold on its training part. Here a single 2k tokeniser per pre-tokeniser, trained on fold 0, only checks that the setup works.

In [ ]:
_, _, fold0_train, _ = FOLDS[0]
fold0_texts = D.train["text"].iloc[fold0_train].tolist()

ours = [
    tok.TokenizerView.from_local(
        tok.train_bpe(fold0_texts, 2000, isolate_punct=isolate), tok.bpe_name(2000, isolate)
    )
    for isolate in (True, False)
]
views = ours + pretrained
display(tok.describe(views))

#### How the tokenisers split text

`pieces` shows each token as the slice of text it covers (for byte-level tokenisers, a character split over several byte tokens appears once per token); `tokens` shows the tokeniser's own strings (`▁` and `Ġ` mark a preceding space). `roberta-bpe`'s offsets leave out the leading space, so its pieces have none.

**Takeaway (smoke test, fold 0, 2k):** our BPE keeps — ’ “ ” as separate tokens, and the space before an isolated punctuation mark becomes its own `▁` token; `µ` and `…` never occur in this training fold, so they become `[UNK]`. `qwen3-bpe` fuses punctuation with neighbouring words (`—it`, `-known`, `”.`), and `roberta-bpe` splits each curly quote into two byte tokens.

In [ ]:
example = "Language isn’t merely a tool—it’s a lens: well-known; “quoted”."
with pd.option_context("display.max_colwidth", None):
    for text in [PROBE, example]:
        display(tok.segmentation_table(views, text))

### Evaluation on CV folds

For each of the 5 folds (repeat 0), our BPE is trained on the fold's training part only, once per configuration (5 vocab sizes × punctuation isolated / vanilla Metaspace). Every tokeniser, ours and the two pretrained ones, is then scored on the validation part: prompts it has never seen, as with Kaggle's test set. Our BPEs are also scored on their own training texts, to measure how much they fit the training prompts. Results are the mean ± SD over folds.

Metrics (`tok.evaluate_tokenizer`):

- **UNK rate**: UNK tokens / all tokens, plus the share of responses with at least one UNK.
- **Fertility**: tokens per whitespace word (as in `shared.tokenization.tokenizer_stats`); lower = fewer splits.
- **Whole-word rate**: share of words (runs of letters, so `isn’t` = `isn` + `t`) covered by exactly one token, overall and by how often the word occurs in the training fold.
- **Cue preservation**: for each cue character (— – ’ ‘ “ ” " ' ; : - …), the share of occurrences that survive encoding and decoding (`cue_roundtrip`) and the share that are *distinct*, i.e. every token covering them covers only that character plus whitespace (`cue_distinct`). A character split into several byte-level tokens still counts as distinct.

In [ ]:
cv = tok.cv_evaluation(D.train["text"], FOLDS, pretrained=pretrained)
for name, table in cv.items():
    table.to_csv(RESULTS_DIR / f"tok_cv_{name}.csv", index=False)

#### Results per tokeniser

**Takeaway:** UNK is only a sanity check. Our BPE produces an UNK in about 1% of validation responses (0.01% of tokens): rare characters that occur in no training-fold text (accented letters such as è or é, Greek letters, superscripts and subscripts, symbols such as `_`, `→`, `°`). The byte-level pretrained tokenisers never do. The pretrained tokenisers need about 1.29 tokens per word; ours needs 8k merges to get down to 1.50.

In [ ]:
cv_mean, cv_sd = tok.fold_summary(cv["summary"])
display(tok.format_mean_sd(cv_mean, cv_sd, {"vocab_size": 0, "unk_rate": 5}))

#### Fertility vs. vocabulary size

**Takeaway:** there is no sharp elbow. Each doubling of the vocabulary buys less (−0.48, −0.39, −0.29, −0.22 tokens per word from 500 to 8k), while the gap between training-fold and validation-fold fertility grows from 0.6% to 9% of the training value: larger vocabularies increasingly fit words of the training prompts. Isolating punctuation costs only 0.03–0.04 tokens per word compared with vanilla Metaspace.

In [ ]:
tok.plot_fertility(
    cv["summary"],
    title="Fertility on validation folds (our BPE vs. pretrained)",
    save_path=TOK_GRAPHS_DIR / "fertility_vs_vocab.png",
);

#### Whole-word rate by word frequency

**Takeaway:** frequent words (100+ occurrences in the training fold) are kept whole by every tokeniser from 2k on. The difference is in rare words: the pretrained tokenisers keep about 45% of words never seen in the training fold and 78–79% of words seen 1–9 times, while ours keeps 1–2% of unseen words at any vocab size; new words always fall into pieces. Words seen 10–99 times are where our vocab size matters: 34% whole at 2k, 71% at 4k, 99% at 8k.

In [ ]:
tok.plot_whole_word(
    cv["whole_word"],
    title="Whole-word rate on validation folds, by frequency in the training fold",
    save_path=TOK_GRAPHS_DIR / "whole_word_by_frequency.png",
);

#### Cue preservation

Share of each cue character's occurrences kept as a distinct token (pooled over folds; column labels give the number of occurrences). `‘` and `…` never occur in the training split and are left out.

**Takeaway:** with punctuation isolated, our BPE keeps every cue as its own token at every vocab size. Without isolation, merges absorb punctuation as the vocabulary grows (vanilla 8k: em-dash 32%, hyphen 25%, apostrophes 6–8%). `roberta-bpe` keeps — : - ; ’ “ ” separate but attaches the straight apostrophe to contractions (`'s`, 4% distinct), because the GPT-2 pre-tokeniser splits off `'s`, `'t`, ... only for the straight apostrophe. `qwen3-bpe` attaches one punctuation character to the following word (`—it`, `-known`, `’s`): em-dash 66%, hyphen 6%, apostrophes 4%, `”` 55%. Every tokeniser round-trips every cue character. "Not distinct" does not mean lost: `’s` and `'s` are still different tokens, so quote style stays visible, but tied to the word rather than as a countable mark.

In [ ]:
from eda import plot_model_heatmap

cue_tokenizers = [
    tok.bpe_name(2000),
    tok.bpe_name(8000),
    tok.bpe_name(2000, False),
    tok.bpe_name(8000, False),
    *tok.PRETRAINED,
]
plot_model_heatmap(
    tok.cue_table(cv["cues"], cue_tokenizers),
    scale="global",
    cmap="Blues",
    title="Cue characters kept as distinct tokens (validation folds)",
    cbar_label="share of occurrences",
    save_path=TOK_GRAPHS_DIR / "cue_distinct.png",
)
print("cue round trip (share of occurrences surviving encode + decode):")
display(tok.cue_table(cv["cues"], cue_tokenizers, value="roundtrip").round(3))

#### Choosing our vocab size

No downstream test is used here, so the choice follows the rule in the plan: the smallest vocab size past the elbow of the fertility curve, provided (i) the training–validation fertility gap stays small, (ii) the whole-word rate on rare words does not collapse, and (iii) cue preservation stays at 1.0. Punctuation isolation is chosen over vanilla Metaspace on criterion (iii): it keeps every cue distinct at a cost of 0.03–0.04 tokens per word.

`fertility_drop` is the gain from doubling the vocabulary; `gap_pct` is the training–validation gap as % of training fertility.

In [ ]:
display(tok.vocab_tradeoff(cv["summary"], cv["whole_word"]).round(3))

#### Decision: our BPE at 2k and 4k, plus 8k for comparison (punctuation isolated)

The fertility curve has no sharp elbow, so the rule leaves a range rather than one size:

- **500 and 1k** split even frequent words: 2.4–2.9 tokens per word, and at 1k only 4% of words seen 10–99 times are kept whole.
- **2k** (gap 2.7%) generalises best to new prompts; **4k** (gap 5.6%) is the middle ground, with 67% of words kept whole. These are the chosen sizes.
- **8k** is handed over as a **comparison condition**: at 1.50 tokens per word it is the closest we get to the pretrained tokenisers (about 1.29), so it separates "trained on our data vs. pretrained" from "finer vs. coarser pieces". It is not preferred on its own: it reaches that ratio mostly through words of the training prompts (9% training–validation gap; only 2% of unseen words kept whole vs. about 45% for the pretrained tokenisers).

All three keep every cue character as a distinct token. Downstream accuracy on the same folds decides between them, which also answers the vocab-size question empirically.

#### Final tokenisers

`tok.final_bpe` trains each handed-over configuration on the whole training split (2,159 responses) and caches it in `cache/` (gitignored): anyone rebuilds it from code on first use, and training is deterministic (the same texts, in any order, give an identical tokeniser). The second table compares all tokenisers on the CV metrics (fold mean ± SD of the per-fold tokenisers with the same settings).

In [ ]:
final = [
    tok.TokenizerView.from_local(tok.final_bpe(vs), tok.bpe_name(vs))
    for vs in tok.HANDOFF_VOCAB_SIZES
]
display(tok.describe(final + pretrained))

compare = [v.name for v in final] + tok.PRETRAINED
metrics = [
    "unk_rate",
    "docs_with_unk",
    "fertility",
    "fertility_gap",
    "whole_word_rate",
    "cue_roundtrip",
    "cue_distinct",
]
display(
    tok.format_mean_sd(cv_mean.loc[compare, metrics], cv_sd.loc[compare, metrics], {"unk_rate": 5})
)
with pd.option_context("display.max_colwidth", None):
    display(tok.segmentation_table(final + pretrained, example)[["n_tokens", "pieces"]])

#### Hand-off: shared registry

Our three BPEs are registered in `shared.tokenization.TOKENIZERS` as `ours-bpe-2k`, `ours-bpe-4k` and `ours-bpe-8k` (listed in `OURS_TOKENIZERS`), so teammates load them by name like the pretrained ones: `get_tokenizer` builds them through `tok.final_bpe` on first use. The check below confirms that the registry gives the same token ids as `tok.final_bpe`, and runs the shared `tokenizer_stats` on the training split.

**Takeaway:** with a 512-token limit (as in the neural track), `ours-bpe-2k` would truncate about 1% of training responses (the longest needs 585 tokens); `ours-bpe-4k` peaks at 506 tokens, just under the limit with the 2 special tokens, and the others stay well below it. On unseen prompts our BPEs need slightly more tokens (see the CV fertility), so the truncation share would be a little higher. Note that these statistics are on texts the final tokenisers were trained on; the CV table above is the fair comparison.

In [ ]:
from shared.tokenization import (
    OURS_TOKENIZERS,
    USED_TOKENIZERS,
    tokenize,
    tokenizer_stats,
)

assert OURS_TOKENIZERS == [v.name for v in final]
sample = D.train["text"].tolist()
assert all(tokenize(sample, v.name) == [e.ids for e in v.encode(sample)] for v in final)
print("registry token ids == tok.final_bpe token ids for", OURS_TOKENIZERS)
display(tokenizer_stats(sample, names=OURS_TOKENIZERS + USED_TOKENIZERS))

### Reflections — Step 2

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 3. Document representations (features)

**What they ask:** think about how to turn documents into features. The tutorial used token occurrences (indicators) and frequencies (bag-of-words). Can we think of other ways?

**Rubric (checklist):** compare **the two representation spaces from the tutorial (token indicators and bag-of-words) plus one more of our choice**.

**Oral-assessment points** (slide: _features_):
- What does each representation capture, and what does it throw away (order, length, frequency)?
- Candidates for the third representation: TF-IDF, character n-grams, word n-grams, stylometric features (sentence length, punctuation rates, markdown usage), or combinations. Why this one for _LLM authorship_?
- Dimensionality / sparsity, and how the choice interacts with the classifier.

Helpers: `part1.clslib.TokenIndicator` (indicator encoder); bag-of-words and the third encoder are ours to write as `part1.clslib.Encoder` subclasses.

### Reflections — Step 3

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 4. Architectures

**What they ask:** the tutorial used logistic regression (log-linear) and a shallow neural network. Experiment with different architectures and their hyperparameters (e.g. more layers, different activation functions, dropout rates).

**Rubric (checklist):** compare **log-linear, single-hidden-layer MLP, plus one more architecture of our choice** (e.g. a modification of the MLP).

**Oral-assessment points** (slide: _architectures_):
- What does the MLP add over the log-linear model, and does this dataset size (~2.7k docs, ~18 classes) support it?
- Why our third architecture, and what hypothesis does it test?
- Which hyperparameters did we tune, how (on what data, to avoid tuning on the evaluation folds), and why those values?

Helpers: `part1.loglin.fit`, `part1.smlp.SparseMLP`, `part1.smlp.train_neural_network_model`, `part1.experiment.cross_validation_experiment(..., model_option="logistic" | "mlp")`.

### Reflections — Step 4

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 5. Evaluation

**What they ask:** evaluate the models. Was the evaluation method adequate for our new experimental setup? Do we want to assess other aspects of performance?

**Rubric (checklist):** comparisons must be **based on cross-validation and a relevant metric** (accuracy or other).

**Oral-assessment points** (slide: _data and metrics_ / _results_):
- Is accuracy appropriate here (balanced classes?) or should we also report macro-F1 / per-class F1 / confusion matrix?
- **Leakage:** `cross_validation_split` shuffles texts at random, so answers to the same prompt land in both train and validation. Do we need grouped folds by `prompt_id`? Compare both and explain the difference.
- How many folds and seeds, and why? (Here: `NUM_FOLDS` folds × `SEEDS`.)
- Other aspects: training/inference time, calibration, which models get confused with each other.

In [ ]:
# from collections import OrderedDict

# # One entry per system, e.g. "logistic_indicator" -> list over seeds of cross_validation_experiment results
# results = OrderedDict()


# def run_cv(name, encoder, model_option, **kwargs):
#     """Run k-fold CV once per seed, seeding right before each split + training run."""
#     results[name] = []
#     for seed in SEEDS:
#         set_seed(seed)
#         splits = part1.experiment.cross_validation_split(pairs, k=NUM_FOLDS, seed=seed)
#         set_seed(seed)
#         res = part1.experiment.cross_validation_experiment(
#             splits,
#             outcome_space,
#             encoder,
#             model_option,
#             save_to=str(RESULTS_DIR / name / f"seed-{seed}"),
#             **kwargs,
#         )
#         results[name].append(res)
#     per_seed = [np.mean(list(r["accuracies"].values())) for r in results[name]]
#     mean, ci = summarise_runs(per_seed)
#     print(f"{name}: acc {mean:.4f}  95% CI {ci}")


# # TODO: e.g.
# # run_cv("logistic_indicator", indicator_enc, "logistic", loglin_args=LOGLIN_ARGS)
# # run_cv("mlp_indicator", indicator_enc, "mlp", smlp_args=SMLP_ARGS)

### Reflections — Step 5

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## 6. Analysis: which system is best?

**What they ask:** analyse the models we assessed and decide which is best.

**Rubric (checklist):** use **Bayesian estimation to analyse results relative to a region of practical equivalence (ROPE)**, and **motivate the ROPE size** ourselves.

**Oral-assessment points** (slide: _results_, plus _extra_):
- What do P(A better), P(equivalent), P(B better) mean, and how do they depend on the ROPE?
- Why our ROPE size (e.g. what accuracy difference would matter in practice for this task)?
- Is the best system clearly better, or practically equivalent to a simpler one? If equivalent, which would we pick and why?
- Error analysis: which LLMs get confused (confusion matrix), and do the errors make sense (same family / provider)?
- Extra slide ideas: grouped-vs-random folds, ensembles, feature inspection (most predictive tokens per model).

Helpers: `part1.analysis.make_data_frame`, `plot_box_and_points`, `BenavoliCorrelatedT(sys_a, sys_b, runs=len(SEEDS)).plot(rope)` / `.probs(rope)`.

### Reflections — Step 6

> _Write here: what we did, why (vs. reasonable alternatives), pros/cons, surprises, open questions for the TA._

-

## AI-assistance & sources log

We must disclose this in the blog post. Keep a running list: what was delegated to an assistant, and which external sources/code were used.

| Date | Who | Tool / source | What for |
|---|---|---|---|
| 2026-11-02 | Cecce | Claude Code | Notebook skeleton (this structure + reproducibility cell) |